# Tremor — NVIDIA GPU acceleration benchmark (cuDF vs pandas)

**This is the acceleration evidence for the submission.** It times Tremor's heavy stage — scoring + aggregating *millions* of comments — on CPU (pandas) and on an NVIDIA GPU (cuDF, via `cudf.pandas`). Same code, two engines.

### Before you run
1. **Runtime → Change runtime type → Hardware accelerator = GPU (T4 is fine)**, then Save.
2. Run every cell top to bottom (Runtime → Run all).
3. It takes ~3 minutes total. At the end, **download `benchmark_results.json`** and commit it to the repo — the dashboard reads it to show the speedup banner.

In [ ]:
# 1) Confirm we actually have a GPU
!nvidia-smi

In [ ]:
# 2) Install the NVIDIA acceleration layer (cuDF). ~1-2 min.
!pip install -q cudf-cu12 --extra-index-url=https://pypi.nvidia.com
import cudf; print('cuDF', cudf.__version__, 'ready')

In [ ]:
# 3) Get the Tremor code (pre-filled with your repo; change only if you fork/rename).
REPO_URL = 'https://github.com/nayanikaprusty518-hue/tremor.git'
!rm -rf tremor && git clone -q $REPO_URL tremor
%cd tremor
#  --- Repo not reachable? Instead: click the folder icon on the left, upload
#      tremor_core.py, generate_data.py and benchmark.py, and skip this cell. ---

In [ ]:
# 4) Generate a large dataset so the speedup is honest and dramatic (5,000,000 comments).
#    Bump to 10_000_000 if you want an even bigger number.
!python generate_data.py --rows 5000000 --out big.parquet

In [ ]:
# 5) CPU baseline (plain pandas). This is the SAME code as the GPU run below.
!python benchmark.py --data big.parquet --out benchmark_results.json

In [ ]:
# 6) GPU run — identical code, one prefix: `python -m cudf.pandas`.
#    pandas calls now execute on the NVIDIA GPU via cuDF.
!python -m cudf.pandas benchmark.py --data big.parquet --out benchmark_results.json

In [ ]:
# 7) The headline result
import json
r = json.load(open('benchmark_results.json'))
print(json.dumps(r, indent=2))
if r.get('speedup'):
    print(f"\n>>> NVIDIA GPU processed {r['cpu']['rows']:,} comments "
          f"{r['speedup']}x faster than CPU "
          f"({r['cpu']['process_s']}s -> {r['gpu']['process_s']}s) <<<")

### Last step
Download **`benchmark_results.json`** (files panel on the left → right-click → Download) and commit it to the repo root. The Streamlit dashboard automatically shows the speedup banner when this file is present.